## Generate Phase 2 Data

In [2]:
import pandas as pd
import numpy as np
np.random.seed(42)

# ---------------------------------------------------
# 1. DISPATCH RULES
# ---------------------------------------------------

In [3]:
def generate_dispatch_rules():
    data = [
        ['R1', 'EDD', 1, 0, 0, 0, 0],
        ['R2', 'SPT', 0, 0, 0, 0, 1],
        ['R3', 'PRIORITY', 0, 1, 0, 0, 0],
        ['R4', 'HYBRID', 0.4, 0.3, 0.1, 0.2, 0]
    ]

    cols = [
        'rule_id', 'rule_name',
        'weight_due_date', 'weight_priority',
        'weight_setup', 'weight_slack',
        'weight_processing_time'
    ]

    df = pd.DataFrame(data, columns=cols)
    df.to_csv('dispatch_rules.csv', index=False)
    return df


# ---------------------------------------------------
# 2. MACHINE STRATEGY
# ---------------------------------------------------

In [4]:
def generate_machine_strategy(machines, rules):
    df = pd.DataFrame({
        'machine_id': machines['machine_id'],
        'rule_id': np.random.choice(rules['rule_id'], size=len(machines))
    })

    df.to_csv('machine_strategy.csv', index=False)
    return df

# ---------------------------------------------------
# 3. JOB FEATURES
# ---------------------------------------------------

In [9]:
def generate_job_features(orders, routing):
    df = routing.merge(orders, on='product_id')

    df['processing_time'] = df['proc_time_min']
    df['remaining_time'] = df.groupby('order_id')['proc_time_min'].transform('sum')
    df['slack_time'] = (
        (pd.to_datetime(df['due_date']) - pd.to_datetime(df['order_date']))
        .dt.total_seconds() / 3600
        - df['remaining_time']
    )

    features = df[[
        'order_id',
        'operation_seq',
        'due_date',
        'remaining_time',
        'slack_time',
        'priority',
        'processing_time'
    ]]

    features.to_csv('job_features.csv', index=False)
    return features


# ---------------------------------------------------
# 4. UPDATE ORDERS
# ---------------------------------------------------

In [5]:
def update_orders(orders):
    orders = orders.copy()

    orders['release_time'] = orders['order_date']
    orders['penalty_per_hour'] = np.random.randint(10, 100, size=len(orders))

    orders.to_csv('orders_updated.csv', index=False)
    return orders

# ---------------------------------------------------
# 5. UPDATE ROUTING
# ---------------------------------------------------

In [6]:
def update_routing(routing):
    routing = routing.copy()

    routing['operation_type'] = np.random.choice(
        ['discrete', 'batch'],
        size=len(routing)
    )

    routing.to_csv('routing_phase2.csv', index=False)
    return routing


# ---------------------------------------------------
# 6. SCENARIO CONFIG
# ---------------------------------------------------

In [7]:

def generate_scenario_config():
    data = [
        ['objective', 'minimize_tardiness'],
        ['enable_setup_optimization', 1],
        ['enable_buffer_constraints', 1],
        ['enable_dispatch_rules', 1]
    ]

    df = pd.DataFrame(data, columns=['config_key', 'value'])
    df.to_csv('scenario_config.csv', index=False)
    return df

# ---------------------------------------------------
# MAIN
# ---------------------------------------------------

In [10]:

def main():
    print("=== Phase 2 Data Generation ===")

    # Load Phase 1.5 outputs
    machines = pd.read_csv('machines_updated.csv')
    orders = pd.read_csv('orders.csv')
    routing = pd.read_csv('routing_updated.csv')

    # Generate
    rules = generate_dispatch_rules()
    generate_machine_strategy(machines, rules)
    generate_job_features(orders, routing)
    update_orders(orders)
    update_routing(routing)
    generate_scenario_config()

    print("\nGenerated:")
    print(" - dispatch_rules.csv")
    print(" - machine_strategy.csv")
    print(" - job_features.csv")
    print(" - orders_updated.csv")
    print(" - routing_phase2.csv")
    print(" - scenario_config.csv")


if __name__ == "__main__":
    main()

=== Phase 2 Data Generation ===

Generated:
 - dispatch_rules.csv
 - machine_strategy.csv
 - job_features.csv
 - orders_updated.csv
 - routing_phase2.csv
 - scenario_config.csv
